# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring.** I'm choosing this because the starter pipeline already gives me a transparent baseline to test against, not just a black box to trust. Testing it directly on the starter data showed the current hand-written rule (`stale_visible_page`: not updated in 180+ days AND 500+ impressions) only flags 17 pages out of 30,000 — far too narrow to be the team's real triage tool. That gap between what the rule catches and what the label says is declining (54.2% of all pages) is exactly the kind of mismatch a ranked, learned score should help close.

In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
stale_visible = ((df.days_since_last_update >= 180) & (df.impressions_90d >= 500)).sum()
declining_rate = (df.trend_direction == "down").mean()
print(f"stale_visible_page rule flags: {stale_visible} / {len(df)} pages")
print(f"trend_direction == 'down' rate: {declining_rate:.1%}")


## 2. The question: decision, action, cost of a wrong call

**Decision:** which pages should a content editor review and refresh first, out of thousands that could plausibly need it. **Who acts:** the content/SEO editor with limited weekly refresh capacity (this maps to a Precision@20 or Precision@50 problem, matching how many pages a reviewer can actually check). **Cost of a wrong call:** two kinds. A false positive wastes an editor's limited hours on a page that wasn't actually declining. A false negative lets a genuinely declining, high-traffic page keep losing visibility unnoticed, which is more costly the more impressions it carries. Given only 17 pages meet the strictest rule while 54% of the dataset is labeled "down," the real risk right now is under-flagging, not over-flagging.

In [ ]:
# No additional numbers needed here — see Section 3 for supporting data.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset support this lane:

In [ ]:
d = df[df.avg_position > 0]  # avg_position == 0 means "no data," not rank 0 — excluded
ctr_by_tier = d.groupby("position_tier")["ctr"].mean().reindex(["top_3","page_1","striking","page_3_5","deep"])
print("Mean CTR by position tier (avg_position>0 only):")
print(ctr_by_tier.round(3))

print()
print(f"Pages labeled 'down' (proxy label): {declining_rate:.1%} of {len(df)}")
print(f"Hand-rule (stale_visible_page) catches only: {stale_visible} pages")


## 4. Careful words: what I can and can't claim

My work will report **observed** and **directional** findings: measured correlations, grouped comparisons, and precision@K scores on a proper client-holdout split. I will not claim that any model **causes** a decline or a recovery, and I will not claim to predict how Google's algorithm works. `trend_direction`/`trend_pct` define the current label, so they are never model inputs. The starter label (`is_declining_label = trend_direction == "down"`) is a current-window proxy, not a future outcome, so any claim I make about it is about reproducing a pattern in the present data, not about forecasting what a page will do next — a stronger future-window label is a goal for later weeks, not this one.

In [ ]:
# Confirms trend_direction / trend_pct are excluded from any future feature set.
leakage_cols = ["trend_direction", "trend_pct"]
print("Columns excluded from model features (label source):", leakage_cols)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.